# Notebook parsing step by step

Declaring the input/output paths, choosing the k-means notebook as the worked example and listing all source notebooks

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt

IPYNB_PATH = Path.cwd().parent / "data" / "raw" / "ipynb"
PARSED_DIR = Path.cwd().parent / "data" / "parsed"
OUT_DIR = PARSED_DIR / "notebooks"
MODUL = "machine_learning"

EXAMPLE = "kmeans_notebook"

notebooks = sorted(IPYNB_PATH.glob("*.ipynb"))
print("Found notebooks:")
for nb_path in notebooks:
    print(" -", nb_path.name)

## The chunk schema

A chunk has the same fields as a slide chunk, so that slides and notebook sections can later live together in one database

In [ ]:
from pydantic import BaseModel

class SlideChunk(BaseModel):
    id: str
    page_numbers: list[int]
    page_reference_path: str
    modul: str
    lecture: str
    title: str
    page_content: str
    context: str | None = None

## Step 0: how is the notebook structured?

A notebook consists of cells. Each cell is either Markdown (text and headings) or Code. Let's look at the order of the cells for the example notebook

In [ ]:
def get_cell_text(cell):
    source = cell.get("source", "")
    if isinstance(source, list):
        return "".join(source)
    return source

notebook = json.loads((IPYNB_PATH / f"{EXAMPLE}.ipynb").read_text(encoding="utf-8"))

cell_types = []
for cell in notebook["cells"]:
    if get_cell_text(cell).strip() == "":
        continue
    cell_types.append(cell["cell_type"])

print(f"The notebook '{EXAMPLE}' has {len(cell_types)} non-empty cells.\n")
print("Order:")
print(cell_types)

colors = []
for cell_type in cell_types:
    if cell_type == "markdown":
        colors.append("tab:blue")
    else:
        colors.append("tab:orange")

plt.figure(figsize=(11, 1.5))
for i in range(len(cell_types)):
    plt.bar(i, 1, color=colors[i])
plt.title(f"{EXAMPLE}: cells (blue = markdown, orange = code)")
plt.yticks([])
plt.xlabel("Cell no.")
plt.show()

## Step 1: the parser function

Now the actual function. It does three things:

1. It walks through all cells and collects the text into a section
2. Every ## heading starts a new section
3. The title section (everything before the first ##) is merged with the first ## section, so that the title and the first section sit in one chunk

Important: headings are only detected in markdown cells. A # in a code cell is a comment and is not treated as a heading. This keeps code cells fully together

Defining parse_notebook: every ## markdown heading starts a new section, code cells are kept whole and the title section is merged into the first one

In [ ]:
def get_notebook_language(notebook):
    metadata = notebook.get("metadata", {})
    language_info = metadata.get("language_info", {})
    return language_info.get("name", "python")

def parse_notebook(path):
    notebook = json.loads(path.read_text(encoding="utf-8"))
    language = get_notebook_language(notebook)
    lecture = path.stem

    notebook_title = None     
    sections = []           
    section_title = None 
    parts = []               

    for cell in notebook["cells"]:
        text = get_cell_text(cell).strip()
        if text == "":
            continue

        if cell["cell_type"] == "code":
            parts.append(f"```{language}\n{text}\n```")
            continue

        first_line = text.splitlines()[0].strip()

        if notebook_title is None and first_line.startswith("#"):
            notebook_title = first_line.lstrip("#").strip()

        if first_line.startswith("## "):
            sections.append((section_title, parts))
            section_title = first_line.lstrip("#").strip()
            parts = [text]
        else:
            parts.append(text)

    sections.append((section_title, parts))

    sections = [(title, p) for title, p in sections if len(p) > 0]

    if len(sections) >= 2:
        first_title = sections[0][0]
        first_parts = sections[0][1] + sections[1][1]
        sections = [(first_title, first_parts)] + sections[2:]

    if notebook_title is None:
        notebook_title = lecture

    chunks = []
    number = 1
    for section_title, parts in sections:
        if section_title is None or section_title == notebook_title:
            title = notebook_title
        else:
            title = f"{notebook_title} — {section_title}"

        chunk = SlideChunk(
            id=f"{lecture}_section_{number}",
            page_numbers=[number],
            page_reference_path="",
            modul=MODUL,
            lecture=lecture,
            title=title,
            page_content="\n\n".join(parts),
            context=None,
        )
        chunks.append(chunk)
        number += 1

    return chunks

## Step 2: apply the parser to the example

Applying the parser to the example notebook and previewing the resulting chunks

In [ ]:
example_chunks = parse_notebook(IPYNB_PATH / f"{EXAMPLE}.ipynb")

print(f"'{EXAMPLE}' was split into {len(example_chunks)} chunks:\n")
for chunk in example_chunks:
    length = len(chunk.page_content)
    preview = chunk.page_content.splitlines()[0][:50]
    print(f"[{chunk.page_numbers[0]}] {chunk.title}")
    print(f"     length: {length} characters | start: {preview!r}")
    print()

## Step 3: did the code blocks stay intact?

A common mistake when splitting is that a code block gets cut in the middle, then the opening or closing ` ``` ` is missing. We quickly check whether the number of ` ``` ` in each chunk is even, every code block has exactly two

Sanity checking that no code fence got split mid block by verifying the count of ` ``` ` per chunk is even

In [ ]:
all_ok = True
for chunk in example_chunks:
    count = chunk.page_content.count("```")
    is_ok = (count % 2 == 0)
    if not is_ok:
        all_ok = False
    print(f"[{chunk.page_numbers[0]}] number of ```: {count} -> {'ok' if is_ok else 'BROKEN'}")

print()
if all_ok:
    print("All code blocks are complete.")
else:
    print("Warning: a code block was cut in the middle!")

## Step 4: how big are the chunks?

For comparison the dashed line shows how big (in characters) the whole notebook used to be as a single chunk

Plotting the per chunk sizes against the old single chunk size, the split yields smaller, fairer units

In [ ]:
numbers = []
lengths = []
for chunk in example_chunks:
    numbers.append(chunk.page_numbers[0])
    lengths.append(len(chunk.page_content))

old_size = sum(lengths)

plt.figure(figsize=(10, 4))
plt.bar(numbers, lengths, color="tab:blue")
plt.axhline(old_size, color="tab:red", linestyle="--", label="before: 1 large chunk")
plt.title(f"{EXAMPLE}: chunk sizes")
plt.xlabel("Chunk no.")
plt.ylabel("Characters")
plt.legend()
plt.show()

## Step 5: parse and save all notebooks

Parsing and saving every source notebook to JSON, the production run of the notebook parser

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

total_chunks = 0
for nb_path in notebooks:
    chunks = parse_notebook(nb_path)

    data = [chunk.model_dump() for chunk in chunks]
    out_path = OUT_DIR / f"{nb_path.stem}.json"
    out_path.write_text(json.dumps(data, indent=2, ensure_ascii=False), encoding="utf-8")

    total_chunks += len(chunks)
    print(f"{nb_path.stem}: {len(chunks)} chunks saved")

print(f"\nDone. {len(notebooks)} notebooks -> {total_chunks} chunks in total.")

## Step 6: before and after

Before: one chunk per notebook (very large). Now: several small sections. The comparison shows, per notebook, how much smaller the largest chunk is now

Comparing, per notebook, the old whole notebook size against the new largest chunk size to quantify the improvement

In [ ]:
names = []
old_sizes = []   
new_sizes = []   

for nb_path in notebooks:
    notebook = json.loads(nb_path.read_text(encoding="utf-8"))
    language = get_notebook_language(notebook)
    all_parts = []
    for cell in notebook["cells"]:
        text = get_cell_text(cell).strip()
        if text == "":
            continue
        if cell["cell_type"] == "code":
            all_parts.append(f"```{language}\n{text}\n```")
        else:
            all_parts.append(text)
    whole_notebook = "\n\n".join(all_parts)

    chunks = parse_notebook(nb_path)
    biggest = max(len(chunk.page_content) for chunk in chunks)

    names.append(nb_path.stem.replace("_notebook", ""))
    old_sizes.append(len(whole_notebook))
    new_sizes.append(biggest)

x = range(len(names))
plt.figure(figsize=(11, 5))
plt.bar([i - 0.2 for i in x], old_sizes, width=0.4, label="old: whole notebook", color="tab:red")
plt.bar([i + 0.2 for i in x], new_sizes, width=0.4, label="new: largest chunk", color="tab:blue")
plt.xticks(list(x), names, rotation=45, ha="right")
plt.ylabel("Characters")
plt.title("Chunk size: old vs. new")
plt.legend()
plt.tight_layout()
plt.show()

## Step 7: a finished chunk in detail

This is the first chunk in full (title + first ## section)

Printing the first finished chunk in full to inspect the title plus first section merge

In [ ]:
chunk = example_chunks[0]

print("id:          ", chunk.id)
print("title:       ", chunk.title)
print("page_numbers:", chunk.page_numbers)
print("length:      ", len(chunk.page_content), "characters")
print("-" * 60)
print(chunk.page_content)

In [ ]:
import json
from pathlib import Path

notebook_chunks_path = Path.cwd().parent / "data" / "parsed_clean" / "notebooks"
notebook_chunks = sorted(notebook_chunks_path.rglob("*.json"))

total = 0
print("Found notebook chunks:")
for chunk_path in notebook_chunks:
    data = json.loads(chunk_path.read_text(encoding="utf-8"))
    n = len(data) if isinstance(data, list) else 1
    print(f" - {chunk_path.name}: {n}")
    total += n

print(f"\n{len(notebook_chunks)} Dateien, {total} Objekte gesamt")